# 第 40 课：NLU——意图识别、槽位抽取与对话状态

ASR 输出文本；NLU 把文本转换成业务可执行的结构，例如 `{intent:set_temperature, value:26, unit:C}`。

<!-- course-upgrade-v2 -->
## 学习导航

| 项目 | 内容 |
|---|---|
| 所属阶段 | 后处理与语义 |
| 建议投入 | 3～5 小时，可分 2～3 次完成 |
| 前置要求 | 完成第 39 课；如果前测低于 2/3，先回看上一课小结 |
| 本课核心 | intent、slot、dialogue state |
| 完成标准 | 能口头解释核心概念；独立完成强化题；从空白重写核心函数 |

高效顺序：**先回答前测 → 预测代码结果 → 再运行 → 修改一个变量 → 关闭答案复现 → 次日回忆。**


<!-- course-upgrade-v2 -->
## 课前诊断（先不要运行代码）

1. 分别用一句话解释：intent、slot、dialogue state。
2. 画出这三个概念之间的输入—输出关系。
3. 写下你最不确定的一点，并给出一个暂时猜测。

自评：答对 0～1 题先复习前置课；答对 2 题可以正常学习；3 题都能讲清楚则直接挑战代码和迁移题。


<!-- course-bridge-v3 -->
## 知识接力：先取回旧知识，再进入本课

### 3 分钟闭卷回忆

在新 Markdown cell 中回答，**不要先翻前文**：raw ASR 与 normalized text 的证据边界；置信度；N-best；会话状态隔离。

- 三项都能用“含义 + 单位/shape + 一个数字例子”回答：进入本课。
- 能回答两项：学习本课，但把缺口记入 `LEARNING_LOG.md`。
- 只能回答零到一项：先回到 [上一课](39_置信度_NBest与语义重排序.ipynb)与[唯一学习路径](../LEARNING_PATH.md)，做一次最小实验；不要靠继续看新术语掩盖断点。

### 本课接口契约

```text
输入：带时间、说话人、候选与置信度的可追溯识别结果
  ↓ 本课要学会的变换、状态或判断
输出：保留原证据、可校准、可拒绝或澄清的文本/语义结果
```

学完后必须能解释：输入的哪个单位/shape/状态若丢失，会让输出“仍能运行却语义错误”。


In [1]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import soundfile as sf
import librosa

def find_root():
    here=Path.cwd().resolve()
    for p in [here,*here.parents]:
        if (p/"pyproject.toml").exists():return p
    raise FileNotFoundError("请从 learn_asr 或 notebooks 目录启动 Jupyter")
ROOT=find_root();plt.rcParams["figure.figsize"]=(11,4)
print("项目根目录:",ROOT)

import torch
import torch.nn as nn
torch.manual_seed(2)
samples=[("打开空调","open_ac"),("开启空调","open_ac"),("把空调打开","open_ac"),("关闭空调","close_ac"),("关掉空调","close_ac"),("把空调关闭","close_ac"),("温度调到二十六度","set_temp"),("设置温度二十四度","set_temp"),("调到二十二度","set_temp")]
labels=sorted(set(y for _,y in samples));chars=sorted(set("".join(x for x,_ in samples)));ci={c:i for i,c in enumerate(chars)}
def bow(text):
    x=torch.zeros(len(chars))
    for c in text:
        if c in ci:x[ci[c]]+=1
    return x
X=torch.stack([bow(x) for x,_ in samples]);Y=torch.tensor([labels.index(y) for _,y in samples])
model=nn.Linear(len(chars),len(labels));opt=torch.optim.Adam(model.parameters(),lr=.08)
for _ in range(300):opt.zero_grad();loss=nn.functional.cross_entropy(model(X),Y);loss.backward();opt.step()
for text in ["请打开空调","空调关掉","温度调到二十三度"]:print(text,labels[model(bow(text)).argmax().item()])

项目根目录: <REPO_ROOT>


请打开空调 open_ac
空调关掉 close_ac
温度调到二十三度 set_temp


这个 bag-of-characters 分类器是教学基线：它没有词序、上下文和 OOV 泛化能力。生产 NLU 会使用预训练 encoder、规则、检索或 LLM，并对领域数据评估。

## 2. 槽位抽取

In [2]:
cn={"二十":20,"二十一":21,"二十二":22,"二十三":23,"二十四":24,"二十五":25,"二十六":26,"二十七":27,"二十八":28}
def parse_command(text):
    intent=labels[model(bow(text)).argmax().item()];slots={}
    for spoken,value in cn.items():
        if spoken in text:slots["temperature"]=value
    return {"intent":intent,"slots":slots}
for s in ["温度调到二十三度","请打开空调"]:print(parse_command(s))

{'intent': 'set_temp', 'slots': {'temperature': 23}}
{'intent': 'open_ac', 'slots': {}}


## 3. 对话状态用于补全省略

用户：“把温度调到二十六度。” 下一句：“还是二十四吧。” 第二句需要上一轮 intent/device 才能补全。状态必须有过期、用户隔离和可审计更新。

## 4. 执行前验证

Schema 校验、权限、范围、设备存在性和确认策略属于业务层。NLU 高置信不代表允许执行危险操作。

## 本课测试

1. intent 与 slot 有何区别？
2. ASR 置信度低但 NLU 置信度高时应信谁？
3. 对话状态为什么必须按用户/会话隔离？
4. 槽位值为什么需要 schema 验证？
5. “删除全部记录”是否应直接执行？

<details><summary>展开参考答案</summary>

1. intent 表示动作类别，slot 是动作参数。2. 综合判断，不能让语义自信掩盖声学不确定。3. 防止上下文串话。4. 防止越界、类型错误和注入。5. 不应，应鉴权并二次确认。

</details>

<!-- course-upgrade-v2 -->
## 强化练习：第 40 课专属题库

请先把答案写进新的 Markdown/Code cell，再展开自评标准。

### A. 基础回忆

1. 不看上文，分别定义 `intent`、`slot`、`dialogue state`。
2. 哪一个量/状态是本课最容易在模块边界丢失的？它的单位和 shape 是什么？
3. 本课至少写出两个“看起来能运行，但结果其实错误”的例子。

### B. 预测与推理

4. 场景：**两个用户共享状态导致槽位串话**。先预测现象，再说明原因，最后给出一项可以验证猜测的指标。
5. 改变本课最关键参数的 0.5×、1×、2×，分别预测准确率、延迟、内存或数值误差怎样变化。
6. 画一张最小数据流图，在每条边标出 dtype、shape、时间单位或概率/代价方向。

### C. 编程与排错

7. 编程任务：**实现 schema 校验和需要确认的策略**。至少加入正常、边界、错误输入三类测试。
8. 故意制造一个 off-by-one、shape、状态未 reset 或数值稳定性错误；记录错误现象和定位过程。
9. 不看本课实现，从空白 cell 重写最核心函数，并用原实现作数值对照。

### D. 迁移与表达

10. 跨课任务：**把 ASR 置信度传入业务决策**。
11. 用 90 秒向没有学过 ASR 的人解释本课；禁止只念术语，必须举一个数字或生活例子。
12. 写出一个生产系统中会监控的指标，以及它异常时优先检查的三处位置。

<details><summary>展开自评标准</summary>

- 每题 0～2 分：0=无法回答；1=方向正确但缺少单位、边界或验证；2=解释完整且能用代码/数字验证。
- 24 分满分：达到 19 分再进入下一课；15～18 分次日重做错题；低于 15 分回看本课图和核心代码。
- 第 4 题必须包含“预测—原因—指标”，第 7～9 题必须真正运行测试，第 10 题必须明确上下游 contract。
- 核心答案至少应正确使用：intent、slot、dialogue state。

</details>


<!-- course-upgrade-v2 -->
## 间隔复习与离场票

### 离场票（现在完成）

- [ ] 我能不用笔记解释 intent、slot、dialogue state。
- [ ] 我能说出本课最常见的错误及其观测现象。
- [ ] 我能从空白重写一个核心函数，并通过至少 3 个测试。
- [ ] 我能说明本课对上一层和下一层接口的影响。

### 复习时间表

- **明天（5 分钟）**：闭卷写出三个核心概念和一个公式/shape。
- **7 天后（15 分钟）**：重做第 4、7、10 题，不运行原答案。
- **30 天后（20 分钟）**：从真实音频或随机张量重新构造一个最小实验。

把错题记录到根目录 `LEARNING_LOG.md`。不要只写“不会”，要写：原判断、证据、正确规则、下次检查动作。
